# LongEval-Sci Catch-Up Notebook

This notebook is the quickest way for teammates to understand the current project state.

Current comparison set:

- 5 base models
- 5 temporal sibling models
- 3 RRF fusion models

Main reports rebuilt here:

- whole-train effectiveness
- monthly growth robustness
- temporal change metrics

In [ ]:
from pathlib import Path
import subprocess
import json

ROOT = Path.cwd()
if ROOT.name == 'scripts':
    ROOT = ROOT.parent

PYTHON = ROOT / '.venv' / 'Scripts' / 'python.exe'
assert PYTHON.exists(), f'Missing interpreter: {PYTHON}'

def run(cmd):
    cmd = [str(x) for x in cmd]
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

def show_text(path_str):
    path = ROOT / path_str
    print(f'===== {path_str} =====')
    print(path.read_text(encoding='utf-8'))

ROOT

## Model Inventory

Base models:

- `official_pyterrier`
- `official_pyterrier_dense`
- `custom_lexical_fulltext`
- `custom_title_abstract_rm3`
- `custom_title_abstract_rerank`

Temporal siblings:

- `official_pyterrier_temporal`
- `official_pyterrier_dense_temporal`
- `custom_lexical_fulltext_temporal`
- `custom_title_abstract_rm3_temporal`
- `custom_title_abstract_rerank_temporal`

Fusion models:

- `rrf_bm25_ta_dense_ta`
- `rrf_bm25_ft_dense_ta`
- `rrf_bm25_ta_bm25_ft_dense_ta`

In [ ]:
important_paths = [
    'outputs/reports/all_models_train_snapshot1/summary.md',
    'outputs/reports/monthly_split/_summary/monthly_comparison.md',
    'outputs/reports/monthly_split/_summary/temporal_change/temporal_change.md',
]

for rel in important_paths:
    path = ROOT / rel
    print(rel, 'OK' if path.exists() else 'MISSING')

## Rebuild the Three Main Reports

These commands do not rerun heavy indexing. They rebuild the current summaries from existing outputs.

In [ ]:
run([PYTHON, 'scripts/build_all_models_train_report.py'])
run([PYTHON, 'scripts/build_monthly_split_summary.py'])
run([PYTHON, 'scripts/build_temporal_change_report.py'])

## Read the Current Summaries

In [ ]:
show_text('outputs/reports/all_models_train_snapshot1/summary.md')

In [ ]:
show_text('outputs/reports/monthly_split/_summary/monthly_comparison.md')

In [ ]:
show_text('outputs/reports/monthly_split/_summary/temporal_change/temporal_change.md')

## Optional Next Commands

If you want to rerun missing pieces rather than just rebuild summaries, these are the main patterns:

Run one baseline on `snapshot-1 train`:

```powershell
python scripts/run_baseline.py --config configs/custom_lexical_fulltext.yaml --train-snapshot1 --qrels-variant dctr
python scripts/run_baseline.py --config configs/custom_title_abstract_rm3.yaml --train-snapshot1 --qrels-variant dctr
python scripts/run_baseline.py --config configs/custom_title_abstract_rerank.yaml --train-snapshot1 --qrels-variant dctr
```

Build RRF fusion runs from existing outputs:

```powershell
python scripts/run_rrf_fusion.py --run-name rrf_bm25_ft_dense_ta --input-run outputs/custom_lexical_fulltext/snapshot-1-train/run.txt --input-run outputs/official_pyterrier_dense/snapshot-1-train/run.txt --train-snapshot1 --qrels-variant dctr
```

Recompute month-based evaluation from existing runs:

```powershell
python scripts/run_snapshot1_monthly_eval.py --config configs/custom_lexical_fulltext.yaml --plan configs/plans/snapshot1_monthly_eval.yaml --qrels-variant dctr --reuse-existing-run
python scripts/build_monthly_split_summary.py
python scripts/build_temporal_change_report.py
```